# F03 SIGISMUND — Réacteur Remotion SVG Neon
## PENTERACT DORN V3 — VIIe Légion

**Rôle** : Rendu Remotion 60fps, courbes SVG neon, timing dynamique, caméra virtuelle.

**Entrées** : `F03_SIGISMUND/IN/plan_de_vol.json` + `IN/*.png`  
**Sorties** : `F03_SIGISMUND/OUT/video_render.mp4`

---
### Séquence de lancement
| Cellule | Action | Modes |
|---|---|---|
| 1 | Montage Drive | tous |
| 2 | Choix du mode | tous |
| 3 | Token GitHub | github |
| 4 | Node.js | tous |
| 5 | Codebase + npm | tous |
| 6 | Render direct/modal | direct, modal |
| 6a | Upload assets → Release | github |
| 6b | Build image Docker (1 fois) | github |
| 6c | Trigger 10 workers | github |
| 6d | Polling statut | github |
| 6e | Téléchargement vidéo | github |
| 7 | CRS_CUSTOS check-in | tous |


In [ ]:
# CELLULE 1 — Montage Google Drive
from google.colab import drive
drive.mount('/content/drive')
print('[OK] Drive monté.')

In [ ]:
# CELLULE 2 — Configuration générale
DRIVE_BASE = '/content/drive/MyDrive/DRIVE_DORN'  # ← adapter si besoin
REPO       = 'kioka8877-ux/DORN'

print('Mode de rendu :')
print('  1 — direct  (1 worker Colab, lent, aucune config)')
print('  2 — modal   (3 workers Modal, nécessite compte Modal)')
print('  3 — github  (10 workers GitHub Actions, recommandé)')
print()

_choix = input('Votre choix [1/2/3] : ').strip()
_modes = {'1': 'direct', '2': 'modal', '3': 'github'}
if _choix not in _modes:
    raise ValueError(f'Choix invalide : {_choix!r} — entrez 1, 2 ou 3')
MODE = _modes[_choix]

print(f'\nDrive base : {DRIVE_BASE}')
print(f'Mode rendu : {MODE}')
print(f'Repo       : {REPO}')
if MODE == 'github':
    print('[INFO] Lance la cellule 3 pour configurer le token GitHub.')


In [ ]:
# CELLULE 3 — Authentification GitHub (MODE = 'github' uniquement)
import os, getpass, base64, subprocess, sys

if MODE != 'github':
    print(f'[SKIP] Mode actuel : {MODE} — cellule non requise.')
else:
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'requests', 'pynacl', '-q'], check=True)
    print('[OK] requests + pynacl installés')

    import requests
    import nacl.public, nacl.encoding

    token = getpass.getpass('Collez votre GitHub PAT puis appuyez sur Entrée : ')
    token = token.strip()
    os.environ['GITHUB_TOKEN'] = token
    print('[OK] Token chargé en mémoire de session')

    headers = {'Authorization': f'Bearer {token}', 'Accept': 'application/vnd.github+json'}
    r = requests.get(f'https://api.github.com/repos/{REPO}', headers=headers)
    if r.status_code != 200:
        raise RuntimeError(f'Token invalide ou permissions insuffisantes (HTTP {r.status_code})')
    print(f'[OK] Accès repo vérifié : {REPO}')

    r_key = requests.get(
        f'https://api.github.com/repos/{REPO}/actions/secrets/public-key',
        headers=headers
    )
    r_key.raise_for_status()
    pub_key_data = r_key.json()
    pub_key_b64  = pub_key_data['key']
    key_id       = pub_key_data['key_id']

    pub_key_bytes = base64.b64decode(pub_key_b64)
    pk  = nacl.public.PublicKey(pub_key_bytes)
    box = nacl.public.SealedBox(pk)
    encrypted     = box.encrypt(token.encode('utf-8'))
    encrypted_b64 = base64.b64encode(encrypted).decode('utf-8')

    r_secret = requests.put(
        f'https://api.github.com/repos/{REPO}/actions/secrets/GH_TOKEN',
        headers=headers,
        json={'encrypted_value': encrypted_b64, 'key_id': key_id}
    )
    if r_secret.status_code in (201, 204):
        print('[OK] Secret GH_TOKEN configuré automatiquement dans le repo DORN')
    else:
        print(f'[WARN] HTTP {r_secret.status_code} — configurez GH_TOKEN manuellement dans Settings > Secrets')

    print('\n[PRÊT] Token actif pour toute la session.')


In [ ]:
# CELLULE 4 — Installation Node.js
import subprocess, sys

result = subprocess.run(['node', '--version'], capture_output=True)
if result.returncode != 0:
    print('Installation Node.js 20.x ...')
    subprocess.run(['bash', '-c',
        'curl -fsSL https://deb.nodesource.com/setup_20.x | bash - && apt-get install -y nodejs'],
        check=True)
else:
    print(f'[OK] Node.js : {result.stdout.decode().strip()}')

r2 = subprocess.run(['npm', '--version'], capture_output=True)
print(f'[OK] npm : {r2.stdout.decode().strip()}')

In [ ]:
# CELLULE 5 — Copie du codebase depuis Drive + npm install
# Les scripts Python sont systematiquement synchronises depuis le repo GitHub
# pour garantir que la version en production est toujours a jour.
import shutil, os, sys, subprocess, requests
from pathlib import Path

codebase_src = Path(DRIVE_BASE) / 'F03_SIGISMUND' / 'CODEBASE'
codebase_dst = Path('/content/F03_CODEBASE')

if codebase_dst.exists():
    shutil.rmtree(codebase_dst)
shutil.copytree(codebase_src, codebase_dst)
print(f'[OK] Codebase copie : {codebase_dst}')

# Ecraser les scripts Python avec la version du repo (source of truth)
_gh_token = os.environ.get('GITHUB_TOKEN', '')
_headers  = {'Authorization': f'token {_gh_token}'} if _gh_token else {}
_raw_base = f'https://raw.githubusercontent.com/{REPO}/main/F03_SIGISMUND/CODEBASE'
for _script in ['drn_f03_sigismund.py', 'drn_f03_gh_trigger.py']:
    _r = requests.get(f'{_raw_base}/{_script}', headers=_headers)
    _r.raise_for_status()
    (codebase_dst / _script).write_bytes(_r.content)
    shutil.copy2(codebase_dst / _script, f'/content/{_script}')
    print(f'[OK] {_script} sync repo → /content/')

# Ajouter le codebase au sys.path pour les imports
if str(codebase_dst) not in sys.path:
    sys.path.insert(0, str(codebase_dst))
print(f'[OK] sys.path mis a jour')

print('npm install ...')
subprocess.run(['npm', 'install', '--no-audit', '--no-fund'], cwd=str(codebase_dst), check=True)
print('[OK] npm install termine')


In [ ]:
# CELLULE 6 — Rendu direct / modal
# Pour MODE = 'github' → passer aux cellules 6a à 6e
import subprocess, sys, os

if MODE == 'github':
    print('[SKIP] Mode github — lance les cellules 6a → 6e.')
else:
    result = subprocess.run(
        [sys.executable, '/content/drn_f03_sigismund.py',
         '--mode',       MODE,
         '--drive-base', DRIVE_BASE],
        capture_output=False
    )
    if result.returncode == 0:
        print('\n[OK] F03 SIGISMUND — RENDU OK')
        print(f'→ Vidéo : {DRIVE_BASE}/F03_SIGISMUND/OUT/video_render.mp4')
    else:
        print('\n[FAIL] F03 SIGISMUND — RENDU FAIL')


---
## GitHub Actions — Étapes 6a → 6e (MODE = 'github' uniquement)

In [ ]:
# CELLULE 6a — Calcul frames + upload assets → GitHub Release
import os, sys, time, json
sys.path.insert(0, '/content/F03_CODEBASE')
from drn_f03_gh_trigger import upload_assets_to_release
from pathlib import Path

if MODE != 'github':
    print(f'[SKIP] Mode actuel : {MODE}')
else:
    F03_IN  = Path(DRIVE_BASE) / 'F03_SIGISMUND' / 'IN'
    F03_OUT = Path(DRIVE_BASE) / 'F03_SIGISMUND' / 'OUT'
    F03_OUT.mkdir(parents=True, exist_ok=True)

    plan_path = F03_IN / 'plan_de_vol.json'
    if not plan_path.exists():
        raise FileNotFoundError(f'plan_de_vol.json introuvable : {plan_path}')

    with open(plan_path) as f:
        plan = json.load(f)

    t = plan['timing']
    reveal = (t['x_range']['end'] - t['x_range']['start']) / t['step_per_frame']
    TOTAL_FRAMES = int(reveal * t.get('complexity_coefficient', 1.0)) + t.get('final_freeze_frames', 180)
    FPS = t.get('fps', 60)
    print(f'[INFO] Total frames : {TOTAL_FRAMES} ({TOTAL_FRAMES / FPS:.1f}s @ {FPS}fps)')

    RUN_ID = f'drn-{int(time.time())}'
    print(f'[INFO] Run ID : {RUN_ID}')

    release_url = upload_assets_to_release(
        f03_in=str(F03_IN),
        run_id=RUN_ID,
        github_token=os.environ['GITHUB_TOKEN'],
        repo=REPO,
    )
    print(f'[OK] Assets uploadés : {release_url}')


In [ ]:
# CELLULE 6b — Build image Docker (run une seule fois)
# Si l'image ghcr.io/kioka8877-ux/dorn-remotion:latest existe déjà → termine en secondes
import os, sys
sys.path.insert(0, '/content/F03_CODEBASE')
from drn_f03_gh_trigger import ensure_docker_image

if MODE != 'github':
    print(f'[SKIP] Mode actuel : {MODE}')
else:
    docker_ok = ensure_docker_image(
        github_token=os.environ['GITHUB_TOKEN'],
        repo=REPO,
        timeout_min=25,
    )
    if not docker_ok:
        raise RuntimeError('[STOP] Build Docker échoué ou timeout. Vérifiez GitHub Actions.')
    print('[OK] Image Docker disponible — prêt pour 6c.')


In [ ]:
# CELLULE 6c — Trigger GitHub Actions (10 workers)
# Une fois lancé, les workers sont indépendants de Colab.
# Tu peux fermer Colab et revenir dans ~5-10 min pour 6d.
import os, sys
sys.path.insert(0, '/content/F03_CODEBASE')
from drn_f03_gh_trigger import trigger_workflow

if MODE != 'github':
    print(f'[SKIP] Mode actuel : {MODE}')
else:
    GH_RUN_ID = trigger_workflow(
        run_id=RUN_ID,
        fps=FPS,
        composition='Main',
        total_frames=TOTAL_FRAMES,
        github_token=os.environ['GITHUB_TOKEN'],
        repo=REPO,
    )
    print(f'[OK] 10 workers GitHub Actions lancés.')
    print(f'[INFO] Tu peux fermer Colab. Reviens dans ~5-10 min et lance 6d.')
    print(f'[INFO] Suivi : https://github.com/{REPO}/actions/runs/{GH_RUN_ID}')


In [ ]:
# CELLULE 6d — Polling statut GitHub Actions
# Si Colab a crashé pendant 6c : relance cellules 1-3, redéfinis GH_RUN_ID
# depuis l'URL GitHub Actions, puis relance cette cellule.
import os, sys
sys.path.insert(0, '/content/F03_CODEBASE')
from drn_f03_gh_trigger import poll_run_status

if MODE != 'github':
    print(f'[SKIP] Mode actuel : {MODE}')
else:
    # Si Colab a crashé, décommenter et renseigner manuellement :
    # GH_RUN_ID = 123456789

    print(f'[POLL] Surveillance du run {GH_RUN_ID}...')
    status = poll_run_status(
        gh_run_id=GH_RUN_ID,
        github_token=os.environ['GITHUB_TOKEN'],
        repo=REPO,
    )
    print(f'\n[OK] Rendu terminé — statut : {status}')
    print('[INFO] Lance maintenant 6e pour télécharger la vidéo.')


In [ ]:
# CELLULE 6e — Téléchargement vidéo finale
import os, sys
sys.path.insert(0, '/content/F03_CODEBASE')
from drn_f03_gh_trigger import download_final_artifact
from pathlib import Path

if MODE != 'github':
    print(f'[SKIP] Mode actuel : {MODE}')
else:
    F03_OUT = Path(DRIVE_BASE) / 'F03_SIGISMUND' / 'OUT'
    output_path = download_final_artifact(
        gh_run_id=GH_RUN_ID,
        run_id=RUN_ID,
        github_token=os.environ['GITHUB_TOKEN'],
        repo=REPO,
        output_dir=str(F03_OUT),
    )
    size_mb = os.path.getsize(output_path) / 1024 / 1024
    print(f'[OK] video_render.mp4 → {output_path} ({size_mb:.1f} MB)')
    print('→ Étape suivante : cellule 7 (CRS_CUSTOS check-in)')


In [ ]:
# CELLULE 7 — Transit CRS_CUSTOS (check-in F03)
# Lancer UNIQUEMENT après RENDU OK (cellule 6 ou 6e)
import shutil, subprocess, sys, os
from pathlib import Path

custos_src = Path(DRIVE_BASE) / 'CRS_CUSTOS.py'
shutil.copy2(custos_src, '/content/CRS_CUSTOS.py')

result = subprocess.run(
    [sys.executable, '/content/CRS_CUSTOS.py',
     '--frigate', 'F03', '--mode', 'check-in', '--drive-base', DRIVE_BASE],
    capture_output=False
)
if result.returncode == 0:
    print('\n[OK] CRS_CUSTOS — F03 check-in OK — Transit autorisé vers F04')
else:
    print('\n[FAIL] CRS_CUSTOS — F03 check-in FAIL')